# Final results after kfold hyperparam. selection

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from utils_table_generator import *

wandb_username = "gbg141"  # Change this to your W&B username if needed
wandb_project = "ProteoFinal"  # Change this to your W&B project name if needed
metric = "mse"  # Change this to the metric you want to extract (e.g., "mae", "mse", etc.)
original_units = True  # Set to True if you want to convert back to original units
csv_filename = "final_results.csv"  # Output CSV filename
save_csv = False  # Set to True if you want to save the grouped results to a CSV file

df = load_results_dataframe(wandb_username, wandb_project, original_units=original_units, metric=metric, csv_filename=csv_filename, save_csv=save_csv)
# df, grouped, best_configs, summary, table = generate_table(df, save_csv=save_csv, csv_filename=csv_filename)

▶ Number of runs fetched from W&B: 3
▶ After building df, df.shape = (3, 165)


In [2]:
columns_to_keep = ["dataset", "model", "test_mae", "checkpoint"]
df = df[columns_to_keep]
pd.set_option('display.max_colwidth', None)
df

,dataset,model,test_mae,checkpoint
0,wgcna,gcn,12.602919,/scratch/lcornelis/outputs/checkpoints/epoch_044-v93.ckpt
1,spearman_correlation,gcn,12.817776,/scratch/lcornelis/outputs/checkpoints/epoch_053-v40.ckpt
2,spearman_correlation,mlp,11.935126,/scratch/lcornelis/outputs/checkpoints/epoch_119-v6.ckpt


In [3]:
df[df["model"]=="mlp"]["checkpoint"]

2    /scratch/lcornelis/outputs/checkpoints/epoch_119-v6.ckpt
Name: checkpoint, dtype: object

In [76]:
import networkx as nx
import numpy as np
import csv
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from hydra.core.global_hydra import GlobalHydra  # Import GlobalHydra explicitly
from topobench.utils.config_resolvers import (
    get_default_metrics,
    get_default_transform,
    get_flattened_feature_matrix_dim,
    get_gatv4_output_dim,
    get_monitor_metric,
    get_monitor_mode,
    get_required_lifting,
    infer_in_channels,
    infer_num_cell_dimensions,
)
from omegaconf import DictConfig, OmegaConf
OmegaConf.register_new_resolver(
    "get_default_metrics", get_default_metrics, replace=True
)
OmegaConf.register_new_resolver(
    "get_default_transform", get_default_transform, replace=True
)
OmegaConf.register_new_resolver(
    "get_flattened_feature_matrix_dim", get_flattened_feature_matrix_dim, replace=True
)
OmegaConf.register_new_resolver(
    "get_gatv4_output_dim", get_gatv4_output_dim, replace=True
)
OmegaConf.register_new_resolver(
    "get_required_lifting", get_required_lifting, replace=True
)
OmegaConf.register_new_resolver(
    "get_monitor_metric", get_monitor_metric, replace=True
)
OmegaConf.register_new_resolver(
    "get_monitor_mode", get_monitor_mode, replace=True
)
OmegaConf.register_new_resolver(
    "infer_in_channels", infer_in_channels, replace=True
)
OmegaConf.register_new_resolver(
    "infer_num_cell_dimensions", infer_num_cell_dimensions, replace=True
)
OmegaConf.register_new_resolver(
    "parameter_multiplication", lambda x, y: int(int(x) * int(y)), replace=True
)
# Clear GlobalHydra instance if already initialized
if GlobalHydra().is_initialized():
    GlobalHydra().clear()

initialize(config_path="../configs", job_name="job")

hydra.initialize()

In [ ]:
model = "mlp"
adj_metric = "spearman_correlation"

cfg = compose(
    config_name="run.yaml",
    overrides=[
        "model=graph/gat",
        "dataset=graph/FTD",
        "dataset=graph/FTD",
        f"model=graph/{model}",
        f"dataset.loader.parameters.adj_metric={adj_metric}",
        "dataset.loader.parameters.adj_thresh=0.50",
        "dataset.loader.parameters.kfold=false",
        "dataset.loader.parameters.num_folds=5",
        "dataset.loader.parameters.fold=0",
        "dataset.dataloader_params.batch_size=16",
        "model.readout.graph_encoder_dim=[512,256]",
        "model.readout.feature_encoder_dim=64",
        "model.readout.fc_dim=[128,64,32]",
        "model.readout.fc_dropout=0.25",
        "model.readout.fc_act=tanh",
        "optimizer.parameters.lr=0.001",
        "dataset.split_params.data_seed=0",
        "trainer.max_epochs=1000",
        "trainer.min_epochs=100",
        "trainer.check_val_every_n_epoch=1",
        "trainer.devices=[7]",
    ], 
    return_hydra_config=False
)

In [78]:
resolved_cfg = OmegaConf.to_container(cfg, resolve=True)
print(resolved_cfg)
cfg = OmegaConf.create(resolved_cfg)

{'task_name': 'train', 'tags': ['dev'], 'train': True, 'test': True, 'ckpt_path': None, 'seed': 42, 'dataset': {'loader': {'_target_': 'topobench.data.loaders.FTDDatasetLoader', 'parameters': {'data_domain': 'graph', 'data_type': 'proteomics', 'data_name': 'FTD', 'dataset_name': 'ftd', 'raw_file_name': 'ALLFTD_dataset_for_nina_louisa_071124_age_adjusted.csv', 'error_protein_file_name': 'bimodal_aptamers_for_removal.xlsx', 'y_val': 'nfl', 'modality': 'csf', 'mutation': ['GRN', 'MAPT', 'C9orf72', 'CTL'], 'sex': ['M', 'F'], 'num_nodes': 7258, 'adj_metric': 'spearman_correlation', 'adj_thresh': 0.5, 'split': 'train', 'wgcna_minModuleSize': 10, 'wgcna_mergeCutHeight': 0.25, 'kfold': False, 'num_folds': 5, 'fold': 0, 'use_weights': False, 'random_state': 42, 'data_dir': '/scratch/lcornelis/data/data_louisa/FTD'}}, 'parameters': {'num_features': 1, 'num_classes': 1, 'task': 'regression', 'loss_type': 'mse', 'monitor_metric': 'mse', 'task_level': 'graph'}, 'split_params': {'learning_setting': 

In [80]:
# loader = instantiate(cfg.dataset.loader)
# dataset, dataset_dir = loader.load()


In [79]:
model = instantiate(
        cfg.model,
        evaluator=cfg.evaluator,
        optimizer=cfg.optimizer,
        loss=cfg.loss,
    )
checkpoint = torch.load("/scratch/lcornelis/outputs/checkpoints/epoch_119-v6.ckpt", map_location="cpu")
model.load_state_dict(checkpoint["state_dict"], strict=False)

<All keys matched successfully>

In [82]:
model.forward()

TypeError: TBModel.forward() missing 1 required positional argument: 'batch'